# 51: Factor Risk Attribution & Multi-Factor Analysis

## Executive Summary

This notebook implements **institutional-grade factor analysis** as used by quantitative asset managers (AQR, Two Sigma, Bridgewater, D.E. Shaw). We decompose portfolio returns into systematic factor exposures, calculate risk contributions, and build multi-factor risk models.

### What Professional Analysts Need:

**1. Factor Exposure Analysis**
- Fama-French 5-factor model regression (Market, SMB, HML, RMW, CMA)
- Carhart 4-factor (adding Momentum)
- Custom factor construction (Quality, Low Vol, Growth)
- Rolling factor beta estimation

**2. Risk Attribution**
- Decompose portfolio risk into factor and idiosyncratic components
- Marginal and component risk contributions
- Factor correlation analysis
- Stress testing factor exposures

**3. Factor Portfolio Construction**
- Factor-tilted portfolios
- Risk parity across factors
- Factor timing signals

**4. Performance Attribution**
- Brinson attribution (allocation vs selection)
- Factor-based attribution
- Alpha decomposition

**Data sources**: `qj.eod.get_historical_prices`, `qj.ff.get_factors`, local pandas portfolio weights

**API:** https://api.quantjourney.cloud

## Run Output

![51_factor_risk_attribution](../plots/51_factor_risk_attribution_output_01.png)

**Prepared by QuantJourney.** Candidate notebook source is kept clean and unexecuted. Generated run artifacts are committed under `plots/` and indexed in `plots/manifest.json`.

In [ ]:
# =============================================================================
# SETUP & CONFIGURATION
# =============================================================================

import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "png"

from quantjourney.sdk import QuantJourneyAPI
from datetime import datetime, timedelta
from scipy import stats
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# API Connection
import os
API_KEY = os.environ.get("QJ_API_KEY", "qj_...")
qj = QuantJourneyAPI(api_key=API_KEY)

# Configuration
pd.set_option('display.float_format', '{:,.4f}'.format)
np.random.seed(42)

print("="*80)
print("FACTOR RISK ATTRIBUTION & MULTI-FACTOR ANALYSIS")
print("Institutional Quantitative Risk Framework")
print("="*80)
print(f"\n✓ Connected to QuantJourney API")
print(f"✓ Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")


---

## Section 1: Portfolio & Factor Data Collection

### Analyst Perspective (AQR Capital Management)

Before conducting factor analysis, we need:

1. **Portfolio Holdings** - Current positions and weights
2. **Historical Returns** - Daily/weekly returns for regression
3. **Factor Returns** - Time series of factor premiums
4. **Risk-Free Rate** - For excess return calculation

**Factor Definitions (Academic Standard):**
- **MKT (Market)**: Market excess return (Rm - Rf)
- **SMB (Size)**: Small minus Big - size premium
- **HML (Value)**: High minus Low book-to-market
- **RMW (Profitability)**: Robust minus Weak operating profitability
- **CMA (Investment)**: Conservative minus Aggressive asset growth
- **MOM (Momentum)**: Winners minus Losers (12-1 month)

We'll also construct custom factors for more granular analysis.

In [ ]:
# =============================================================================
# SECTION 1: DATA COLLECTION
# =============================================================================

print("\n" + "="*80)
print("DATA COLLECTION")
print("="*80)

# 1.1 Define portfolio
print("\n[1.1] Sample Institutional Portfolio")
print("-" * 50)

portfolio = {
    'AAPL': 0.12,   # Large cap tech
    'MSFT': 0.10,   # Large cap tech
    'GOOGL': 0.08,  # Large cap tech
    'JPM': 0.08,    # Financials
    'JNJ': 0.07,    # Healthcare
    'XOM': 0.06,    # Energy
    'PG': 0.06,     # Consumer staples
    'NVDA': 0.08,   # Growth tech
    'UNH': 0.07,    # Healthcare
    'HD': 0.06,     # Consumer discretionary
    'BAC': 0.05,    # Financials
    'CVX': 0.05,    # Energy
    'AMZN': 0.07,   # Large cap growth
    'META': 0.05    # Large cap tech
}

symbols = list(portfolio.keys())
weights = np.array(list(portfolio.values()))

print(f"     Portfolio: {len(symbols)} holdings")
print(f"     Total Weight: {weights.sum()*100:.1f}%")

# Sector mapping
sector_map = {
    'AAPL': 'Technology', 'MSFT': 'Technology', 'GOOGL': 'Technology',
    'JPM': 'Financials', 'BAC': 'Financials',
    'JNJ': 'Healthcare', 'UNH': 'Healthcare',
    'XOM': 'Energy', 'CVX': 'Energy',
    'PG': 'Consumer Staples', 'HD': 'Consumer Discretionary',
    'NVDA': 'Technology', 'AMZN': 'Consumer Discretionary', 'META': 'Technology'
}

portfolio_df = pd.DataFrame({
    'Symbol': symbols,
    'Weight': weights,
    'Sector': [sector_map[s] for s in symbols]
})
print(portfolio_df.to_string(index=False))


In [ ]:
# =============================================================================
# FETCH HISTORICAL RETURNS
# =============================================================================

print("\n[1.2] Fetching Historical Price Data (2 Years)")
print("-" * 50)

end_date = datetime.now()
start_date = end_date - timedelta(days=730)  # 2 years

returns_data = {}
data_loaded = False

# Try to fetch from API
for symbol in symbols + ['SPY']:  # Include SPY as market proxy
    try:
        response = qj.eod.get_historical_prices(
            symbol=symbol,
            start_date=start_date.strftime('%Y-%m-%d'),
            end_date=end_date.strftime('%Y-%m-%d')
        )
        data = response.get('data', response.get('value', response)) if isinstance(response, dict) else response
        if isinstance(data, dict):
            data = data.get(symbol) or data.get(symbol.upper()) or data.get('prices') or data.get('rows') or data.get('results') or data
        if isinstance(data, dict):
            data = [data]
        
        if isinstance(data, list) and len(data) > 100:
            df = pd.DataFrame(data)
            df['date'] = pd.to_datetime(df['date'])
            df = df.sort_values('date').set_index('date')
            price_col = 'adjusted_close' if 'adjusted_close' in df.columns else 'close'
            returns_data[symbol] = pd.to_numeric(df[price_col], errors='coerce').pct_change().dropna()
            print(f"     ✓ {symbol}: {len(returns_data[symbol])} days")
            data_loaded = True
    except Exception as e:
        pass

if not data_loaded or len(returns_data) < len(symbols):
    print("\n     Generating synthetic return data...")
    
    # Generate realistic correlated returns
    n_days = 504  # 2 years of trading days
    dates = pd.date_range(end=end_date, periods=n_days, freq='B')
    
    # Stock characteristics (annual return, vol, beta)
    stock_params = {
        'AAPL': (0.15, 0.28, 1.2), 'MSFT': (0.18, 0.26, 1.1),
        'GOOGL': (0.12, 0.30, 1.15), 'JPM': (0.10, 0.28, 1.3),
        'JNJ': (0.05, 0.18, 0.7), 'XOM': (0.08, 0.30, 1.0),
        'PG': (0.06, 0.16, 0.6), 'NVDA': (0.35, 0.50, 1.8),
        'UNH': (0.12, 0.22, 0.9), 'HD': (0.10, 0.25, 1.1),
        'BAC': (0.08, 0.35, 1.4), 'CVX': (0.07, 0.28, 1.0),
        'AMZN': (0.20, 0.35, 1.3), 'META': (0.25, 0.40, 1.4),
        'SPY': (0.10, 0.18, 1.0)
    }
    
    # Generate market returns first
    mkt_ret, mkt_vol, _ = stock_params['SPY']
    daily_mkt_ret = mkt_ret / 252
    daily_mkt_vol = mkt_vol / np.sqrt(252)
    market_returns = np.random.normal(daily_mkt_ret, daily_mkt_vol, n_days)
    
    returns_data = {'SPY': pd.Series(market_returns, index=dates)}
    
    for symbol, (ret, vol, beta) in stock_params.items():
        if symbol == 'SPY':
            continue
        daily_vol = vol / np.sqrt(252)
        daily_ret = ret / 252
        
        # Beta-adjusted returns + idiosyncratic
        idio_vol = np.sqrt(daily_vol**2 - (beta * daily_mkt_vol)**2)
        if idio_vol < 0:
            idio_vol = 0.01
        
        alpha = daily_ret - beta * daily_mkt_ret
        stock_returns = alpha + beta * market_returns + np.random.normal(0, idio_vol, n_days)
        returns_data[symbol] = pd.Series(stock_returns, index=dates)
    
    print(f"     ✓ Generated {n_days} days of synthetic returns for {len(returns_data)} securities")

# Combine into DataFrame
returns_df = pd.DataFrame(returns_data)
returns_df = returns_df.dropna()

print(f"\n     Final dataset: {len(returns_df)} trading days")
print(f"     Date range: {returns_df.index[0].strftime('%Y-%m-%d')} to {returns_df.index[-1].strftime('%Y-%m-%d')}")


In [ ]:
# =============================================================================
# CREATE FACTOR RETURNS
# =============================================================================

print("\n[1.3] Constructing Factor Returns")
print("-" * 50)

# Try to fetch Fama-French factors from API
try:
    response = qj.ff.get_factors(
        region='US',
        frequency='daily'
    )
    data = response.get('data', response.get('value', response)) if isinstance(response, dict) else response
    if isinstance(data, list) and len(data) > 100:
        factor_df = pd.DataFrame(data)
        factor_df['date'] = pd.to_datetime(factor_df['date'])
        factor_df = factor_df.set_index('date')
        print("     ✓ Fama-French factors loaded from API")
        factors_loaded = True
    else:
        factors_loaded = False
except:
    factors_loaded = False

if not factors_loaded:
    print("     Generating synthetic factor returns...")
    
    # Factor parameters (daily mean, daily vol, market correlation)
    factor_params = {
        'MKT': (0.00040, 0.0115, 1.00),   # Market excess return
        'SMB': (0.00008, 0.0055, 0.10),   # Size premium
        'HML': (0.00005, 0.0060, -0.20),  # Value premium (neg corr recently)
        'RMW': (0.00010, 0.0040, 0.05),   # Profitability
        'CMA': (0.00006, 0.0035, -0.15),  # Investment
        'MOM': (0.00012, 0.0070, -0.05),  # Momentum
        'RF': (0.00018, 0.0001, 0.00)     # Risk-free (T-bill)
    }
    
    n_days = len(returns_df)
    
    # Market factor (correlated with SPY)
    market_factor = returns_df['SPY'].values - factor_params['RF'][0]
    
    factor_data = {'MKT': market_factor}
    
    # Generate other factors with correlations
    for factor, (mean, vol, mkt_corr) in factor_params.items():
        if factor in ['MKT', 'RF']:
            continue
        
        # Factor return = alpha + beta*market + idiosyncratic
        idio_noise = np.random.normal(0, vol * 0.9, n_days)
        factor_returns = mean + mkt_corr * 0.3 * market_factor + idio_noise
        factor_data[factor] = factor_returns
    
    # Risk-free rate
    factor_data['RF'] = np.ones(n_days) * factor_params['RF'][0]
    
    factor_df = pd.DataFrame(factor_data, index=returns_df.index)
    print(f"     ✓ Generated {len(factor_df)} days of factor returns")

print("\n     Factor Statistics (Annualized):")
factor_stats = pd.DataFrame({
    'Mean (%)': factor_df.mean() * 252 * 100,
    'Vol (%)': factor_df.std() * np.sqrt(252) * 100,
    'Sharpe': (factor_df.mean() / factor_df.std()) * np.sqrt(252)
}).round(2)
print(factor_stats.to_string())


---

## Section 2: Single-Stock Factor Exposure Analysis

### Analyst Perspective (Two Sigma Quantitative Research)

**Factor Regression Model:**

For each stock, we estimate factor exposures using time-series regression:

$$R_i - R_f = \alpha_i + \beta_{MKT,i}(R_m - R_f) + \beta_{SMB,i} \cdot SMB + \beta_{HML,i} \cdot HML + \beta_{RMW,i} \cdot RMW + \beta_{CMA,i} \cdot CMA + \epsilon_i$$

**Interpretation:**
- **Alpha (α)**: Excess return not explained by factors (skill)
- **Factor Betas (β)**: Sensitivity to each factor
- **R-squared**: % of variance explained by factors
- **Idiosyncratic Risk**: Residual volatility (diversifiable)

High-quality factor models should explain 60-80% of return variance for diversified portfolios.

In [ ]:
# =============================================================================
# SECTION 2: SINGLE-STOCK FACTOR REGRESSIONS
# =============================================================================

print("\n" + "="*80)
print("SINGLE-STOCK FACTOR EXPOSURE ANALYSIS")
print("="*80)

# Risk-free rate
rf = factor_df['RF'].values

# Factor matrix (exclude RF)
factor_names = ['MKT', 'SMB', 'HML', 'RMW', 'CMA', 'MOM']
X = factor_df[factor_names].values

# Add constant for alpha
X_with_const = np.column_stack([np.ones(len(X)), X])

# Run regressions for each stock
factor_exposures = []

print("\n[2.1] Factor Regression Results")
print("-" * 50)

for symbol in symbols:
    # Excess returns
    y = returns_df[symbol].values - rf
    
    # OLS regression
    try:
        beta, residuals, rank, s = np.linalg.lstsq(X_with_const, y, rcond=None)
        
        # Calculate R-squared
        y_pred = X_with_const @ beta
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - y.mean()) ** 2)
        r_squared = 1 - (ss_res / ss_tot)
        
        # Idiosyncratic vol (annualized)
        idio_vol = np.std(y - y_pred) * np.sqrt(252)
        
        exposure = {
            'Symbol': symbol,
            'Alpha': beta[0] * 252,  # Annualized
            'MKT': beta[1],
            'SMB': beta[2],
            'HML': beta[3],
            'RMW': beta[4],
            'CMA': beta[5],
            'MOM': beta[6],
            'R2': r_squared,
            'Idio_Vol': idio_vol
        }
        factor_exposures.append(exposure)
        
    except Exception as e:
        print(f"     Error for {symbol}: {e}")

exposure_df = pd.DataFrame(factor_exposures)

# Display results
print("\n     FACTOR EXPOSURES (BETAS)")
print("     " + "="*100)
display_cols = ['Symbol', 'Alpha', 'MKT', 'SMB', 'HML', 'MOM', 'R2', 'Idio_Vol']
print(exposure_df[display_cols].round(3).to_string(index=False))


In [ ]:
# =============================================================================
# FACTOR EXPOSURE VISUALIZATION
# =============================================================================

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Market Beta by Stock',
        'Value (HML) vs Size (SMB) Exposure',
        'Alpha vs R-squared',
        'Factor Exposure Heatmap'
    ],
    specs=[[{'type': 'bar'}, {'type': 'scatter'}],
           [{'type': 'scatter'}, {'type': 'heatmap'}]]
)

# 1. Market Beta
colors = ['lime' if b > 1 else 'cyan' for b in exposure_df['MKT']]
fig.add_trace(go.Bar(
    x=exposure_df['Symbol'],
    y=exposure_df['MKT'],
    marker_color=colors,
    name='Market Beta'
), row=1, col=1)
fig.add_hline(y=1.0, line_dash='dash', line_color='yellow', row=1, col=1)

# 2. Value vs Size
fig.add_trace(go.Scatter(
    x=exposure_df['SMB'],
    y=exposure_df['HML'],
    mode='markers+text',
    marker=dict(size=12, color='cyan'),
    text=exposure_df['Symbol'],
    textposition='top center',
    name='Stocks'
), row=1, col=2)
fig.add_hline(y=0, line_dash='dash', line_color='gray', row=1, col=2)
fig.add_vline(x=0, line_dash='dash', line_color='gray', row=1, col=2)

# 3. Alpha vs R2
fig.add_trace(go.Scatter(
    x=exposure_df['R2'],
    y=exposure_df['Alpha'] * 100,
    mode='markers+text',
    marker=dict(
        size=exposure_df['Idio_Vol'] * 100,
        color=exposure_df['MKT'],
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(title='Beta', x=0.45, y=0.2, len=0.3)
    ),
    text=exposure_df['Symbol'],
    textposition='top center',
    name='Stocks'
), row=2, col=1)

# 4. Heatmap
heatmap_data = exposure_df[['MKT', 'SMB', 'HML', 'RMW', 'CMA', 'MOM']].values
fig.add_trace(go.Heatmap(
    z=heatmap_data,
    x=['MKT', 'SMB', 'HML', 'RMW', 'CMA', 'MOM'],
    y=exposure_df['Symbol'],
    colorscale='RdBu',
    zmid=0,
    text=np.round(heatmap_data, 2),
    texttemplate='%{text}',
    showscale=True
), row=2, col=2)

fig.update_layout(
    title=dict(text='Stock-Level Factor Exposures', font=dict(size=20)),
    template='plotly_dark',
    height=800,
    showlegend=False
)

fig.update_xaxes(title_text='SMB Beta', row=1, col=2)
fig.update_yaxes(title_text='HML Beta', row=1, col=2)
fig.update_xaxes(title_text='R-squared', row=2, col=1)
fig.update_yaxes(title_text='Alpha (%)', row=2, col=1)

fig.show()


---

## Section 3: Portfolio Factor Exposure & Risk Decomposition

### Analyst Perspective (Bridgewater Associates)

**Portfolio Factor Exposure:**

Portfolio factor betas are the weighted average of individual stock betas:

$$\beta_{p,f} = \sum_{i=1}^{n} w_i \cdot \beta_{i,f}$$

**Risk Decomposition:**

Total portfolio variance decomposes into:
1. **Factor Risk**: Systematic risk from factor exposures
2. **Idiosyncratic Risk**: Stock-specific risk (diversifiable)

$$\sigma_p^2 = \beta_p' \Sigma_f \beta_p + \sum_{i=1}^{n} w_i^2 \sigma_{\epsilon,i}^2$$

Where $\Sigma_f$ is the factor covariance matrix.

**Risk Contribution:**

Marginal contribution to risk (MCTR) and component contribution (CCTR):
- MCTR = $\frac{\partial \sigma_p}{\partial w_i}$
- CCTR = $w_i \times MCTR_i$

In [ ]:
# =============================================================================
# SECTION 3: PORTFOLIO FACTOR EXPOSURE
# =============================================================================

print("\n" + "="*80)
print("PORTFOLIO FACTOR EXPOSURE & RISK DECOMPOSITION")
print("="*80)

# 3.1 Portfolio-level factor betas
print("\n[3.1] Portfolio Factor Betas")
print("-" * 50)

portfolio_betas = {}
for factor in factor_names:
    portfolio_betas[factor] = np.sum(weights * exposure_df[factor].values)

# Portfolio alpha
portfolio_alpha = np.sum(weights * exposure_df['Alpha'].values)

print(f"     Portfolio Alpha (annualized): {portfolio_alpha*100:.2f}%")
print(f"     Factor Betas:")
for factor, beta in portfolio_betas.items():
    print(f"       {factor:6}: {beta:+.3f}")


In [ ]:
# =============================================================================
# RISK DECOMPOSITION
# =============================================================================

print("\n[3.2] Portfolio Risk Decomposition")
print("-" * 50)

# Factor covariance matrix (annualized)
factor_cov = factor_df[factor_names].cov() * 252

# Portfolio factor exposure vector
beta_vector = np.array([portfolio_betas[f] for f in factor_names])

# Factor variance
factor_variance = beta_vector @ factor_cov.values @ beta_vector

# Idiosyncratic variance (weighted sum of squared idio vols)
idio_vars = exposure_df['Idio_Vol'].values ** 2
idio_variance = np.sum(weights ** 2 * idio_vars)

# Total variance
total_variance = factor_variance + idio_variance
total_vol = np.sqrt(total_variance)

# Risk breakdown
factor_risk_pct = factor_variance / total_variance * 100
idio_risk_pct = idio_variance / total_variance * 100

print(f"     Total Portfolio Volatility: {total_vol*100:.2f}%")
print(f"     Factor Volatility: {np.sqrt(factor_variance)*100:.2f}% ({factor_risk_pct:.1f}% of total)")
print(f"     Idiosyncratic Vol: {np.sqrt(idio_variance)*100:.2f}% ({idio_risk_pct:.1f}% of total)")

# Individual factor risk contributions
print("\n     Factor Risk Contributions:")
factor_risk_contrib = {}
for i, factor in enumerate(factor_names):
    # Marginal contribution
    mctr = (factor_cov.values @ beta_vector)[i] * portfolio_betas[factor] / factor_variance
    # Component contribution
    contrib = portfolio_betas[factor] ** 2 * factor_cov.values[i, i]
    factor_risk_contrib[factor] = contrib
    print(f"       {factor:6}: {np.sqrt(contrib)*100:.2f}% vol ({contrib/factor_variance*100:.1f}% of factor risk)")


In [ ]:
# =============================================================================
# RISK DECOMPOSITION VISUALIZATION
# =============================================================================

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Portfolio Factor Betas',
        'Risk Decomposition',
        'Factor Risk Contribution',
        'Factor Correlation Matrix'
    ],
    specs=[[{'type': 'bar'}, {'type': 'pie'}],
           [{'type': 'bar'}, {'type': 'heatmap'}]]
)

# 1. Portfolio Betas
colors = ['lime' if v > 0 else 'red' for v in portfolio_betas.values()]
fig.add_trace(go.Bar(
    x=list(portfolio_betas.keys()),
    y=list(portfolio_betas.values()),
    marker_color=colors
), row=1, col=1)

# 2. Risk Pie
fig.add_trace(go.Pie(
    labels=['Factor Risk', 'Idiosyncratic Risk'],
    values=[factor_variance, idio_variance],
    marker=dict(colors=['cyan', 'orange'])
), row=1, col=2)

# 3. Factor Risk Contribution
contrib_values = [np.sqrt(v) * 100 for v in factor_risk_contrib.values()]
fig.add_trace(go.Bar(
    x=list(factor_risk_contrib.keys()),
    y=contrib_values,
    marker_color='cyan'
), row=2, col=1)

# 4. Factor Correlation
factor_corr = factor_df[factor_names].corr()
fig.add_trace(go.Heatmap(
    z=factor_corr.values,
    x=factor_names,
    y=factor_names,
    colorscale='RdBu',
    zmid=0,
    text=np.round(factor_corr.values, 2),
    texttemplate='%{text}'
), row=2, col=2)

fig.update_layout(
    title=dict(text='Portfolio Risk Decomposition Dashboard', font=dict(size=20)),
    template='plotly_dark',
    height=700
)

fig.update_yaxes(title_text='Beta', row=1, col=1)
fig.update_yaxes(title_text='Vol Contribution (%)', row=2, col=1)

fig.show()


---

## Section 4: Rolling Factor Analysis & Regime Detection

### Analyst Perspective (D.E. Shaw Systematic Strategies)

**Rolling Factor Exposures:**

Factor exposures are not static. Professional quants track:
- Rolling betas (typically 60-day or 252-day windows)
- Structural breaks in factor relationships
- Regime-dependent exposures

**Regime Detection:**
- Risk-on vs Risk-off regimes
- Factor rotation (e.g., Value vs Growth leadership)
- Volatility regimes

This helps identify when factor relationships break down or strengthen.

In [ ]:
# =============================================================================
# SECTION 4: ROLLING FACTOR ANALYSIS
# =============================================================================

print("\n" + "="*80)
print("ROLLING FACTOR ANALYSIS")
print("="*80)

# 4.1 Rolling 60-day betas
print("\n[4.1] Rolling Factor Betas (60-day window)")
print("-" * 50)

window = 60

# Calculate portfolio returns
portfolio_returns = (returns_df[symbols] * weights).sum(axis=1)
portfolio_excess = portfolio_returns - factor_df['RF']

# Rolling regressions
rolling_betas = {factor: [] for factor in factor_names}
rolling_alpha = []
rolling_r2 = []
rolling_dates = []

for i in range(window, len(portfolio_excess)):
    y = portfolio_excess.iloc[i-window:i].values
    X = factor_df[factor_names].iloc[i-window:i].values
    X_const = np.column_stack([np.ones(window), X])
    
    try:
        beta, _, _, _ = np.linalg.lstsq(X_const, y, rcond=None)
        
        y_pred = X_const @ beta
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - y.mean()) ** 2)
        r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
        
        rolling_alpha.append(beta[0] * 252)  # Annualized
        for j, factor in enumerate(factor_names):
            rolling_betas[factor].append(beta[j+1])
        rolling_r2.append(r2)
        rolling_dates.append(portfolio_excess.index[i])
    except:
        pass

rolling_df = pd.DataFrame(rolling_betas, index=rolling_dates)
rolling_df['Alpha'] = rolling_alpha
rolling_df['R2'] = rolling_r2

print(f"     Rolling analysis: {len(rolling_df)} observations")
print(f"     Beta Statistics (60-day rolling):")
print(rolling_df[factor_names].describe().round(3).to_string())


In [ ]:
# =============================================================================
# ROLLING BETAS VISUALIZATION
# =============================================================================

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[
        'Rolling Market Beta',
        'Rolling SMB (Size) Beta',
        'Rolling HML (Value) Beta',
        'Rolling Momentum Beta',
        'Rolling Alpha',
        'Rolling R-squared'
    ],
    vertical_spacing=0.08
)

# Market Beta
fig.add_trace(go.Scatter(
    x=rolling_df.index, y=rolling_df['MKT'],
    mode='lines', line=dict(color='cyan', width=1.5),
    name='MKT Beta'
), row=1, col=1)
fig.add_hline(y=1.0, line_dash='dash', line_color='yellow', row=1, col=1)

# SMB Beta
fig.add_trace(go.Scatter(
    x=rolling_df.index, y=rolling_df['SMB'],
    mode='lines', line=dict(color='lime', width=1.5),
    fill='tozeroy', fillcolor='rgba(0,255,0,0.1)',
    name='SMB Beta'
), row=1, col=2)
fig.add_hline(y=0, line_dash='dash', line_color='gray', row=1, col=2)

# HML Beta
fig.add_trace(go.Scatter(
    x=rolling_df.index, y=rolling_df['HML'],
    mode='lines', line=dict(color='orange', width=1.5),
    fill='tozeroy', fillcolor='rgba(255,165,0,0.1)',
    name='HML Beta'
), row=2, col=1)
fig.add_hline(y=0, line_dash='dash', line_color='gray', row=2, col=1)

# Momentum Beta
fig.add_trace(go.Scatter(
    x=rolling_df.index, y=rolling_df['MOM'],
    mode='lines', line=dict(color='magenta', width=1.5),
    fill='tozeroy', fillcolor='rgba(255,0,255,0.1)',
    name='MOM Beta'
), row=2, col=2)
fig.add_hline(y=0, line_dash='dash', line_color='gray', row=2, col=2)

# Alpha
fig.add_trace(go.Scatter(
    x=rolling_df.index, y=rolling_df['Alpha'] * 100,
    mode='lines', line=dict(color='lime', width=1.5),
    fill='tozeroy', fillcolor='rgba(0,255,0,0.1)',
    name='Alpha'
), row=3, col=1)
fig.add_hline(y=0, line_dash='dash', line_color='gray', row=3, col=1)

# R-squared
fig.add_trace(go.Scatter(
    x=rolling_df.index, y=rolling_df['R2'] * 100,
    mode='lines', line=dict(color='cyan', width=1.5),
    name='R²'
), row=3, col=2)

fig.update_layout(
    title=dict(text='Rolling Factor Exposures (60-Day Window)', font=dict(size=20)),
    template='plotly_dark',
    height=800,
    showlegend=False
)

fig.update_yaxes(title_text='Alpha (%)', row=3, col=1)
fig.update_yaxes(title_text='R² (%)', row=3, col=2)

fig.show()


---

## Section 5: Factor Attribution & Performance Decomposition

### Analyst Perspective (BlackRock Factor Investing)

**Performance Attribution:**

Decompose portfolio returns into factor contributions:

$$R_p = \alpha + \sum_{f=1}^{k} \beta_f \cdot R_f + \epsilon$$

**Attribution Components:**
1. **Alpha**: Manager skill (stock selection)
2. **Factor Returns**: Compensation for systematic risks
3. **Residual**: Unexplained returns

This analysis answers: "Was performance driven by skill or factor timing?"

In [ ]:
# =============================================================================
# SECTION 5: PERFORMANCE ATTRIBUTION
# =============================================================================

print("\n" + "="*80)
print("FACTOR PERFORMANCE ATTRIBUTION")
print("="*80)

# 5.1 Full period attribution
print("\n[5.1] Performance Attribution (Full Period)")
print("-" * 50)

# Portfolio total return (annualized)
total_return = (1 + portfolio_returns).prod() ** (252 / len(portfolio_returns)) - 1

# Factor contributions
factor_contributions = {}
for factor in factor_names:
    factor_return = factor_df[factor].mean() * 252  # Annualized factor return
    contribution = portfolio_betas[factor] * factor_return
    factor_contributions[factor] = contribution

# Sum of factor contributions
total_factor_contrib = sum(factor_contributions.values())

# Risk-free contribution
rf_contrib = factor_df['RF'].mean() * 252

# Alpha + residual
alpha_contrib = total_return - total_factor_contrib - rf_contrib

print(f"     Portfolio Total Return: {total_return*100:+.2f}%")
print(f"")
print(f"     Attribution Breakdown:")
print(f"       Risk-Free:            {rf_contrib*100:+.2f}%")
for factor, contrib in factor_contributions.items():
    print(f"       {factor:20}:  {contrib*100:+.2f}% (β={portfolio_betas[factor]:.3f})")
print(f"       Alpha/Residual:       {alpha_contrib*100:+.2f}%")
print(f"       " + "="*40)
print(f"       Total:                {total_return*100:+.2f}%")


In [ ]:
# =============================================================================
# CUMULATIVE ATTRIBUTION
# =============================================================================

print("\n[5.2] Cumulative Factor Attribution")
print("-" * 50)

# Daily attribution
daily_attribution = pd.DataFrame(index=returns_df.index)
daily_attribution['Portfolio'] = portfolio_returns
daily_attribution['RF'] = factor_df['RF']

for factor in factor_names:
    daily_attribution[f'{factor}_contrib'] = portfolio_betas[factor] * factor_df[factor]

daily_attribution['Alpha_contrib'] = (
    portfolio_returns - factor_df['RF'] - 
    sum(daily_attribution[f'{f}_contrib'] for f in factor_names)
)

# Cumulative returns
cum_attribution = pd.DataFrame(index=returns_df.index)
cum_attribution['Portfolio'] = (1 + portfolio_returns).cumprod() - 1

for factor in factor_names:
    cum_attribution[factor] = (1 + daily_attribution[f'{factor}_contrib']).cumprod() - 1

cum_attribution['Alpha'] = (1 + daily_attribution['Alpha_contrib']).cumprod() - 1

print(f"     Cumulative Attribution (end of period):")
print(cum_attribution.iloc[-1].round(4).to_string())


In [ ]:
# =============================================================================
# ATTRIBUTION VISUALIZATION
# =============================================================================

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Return Attribution Breakdown',
        'Factor Contribution Bar',
        'Cumulative Factor Attribution',
        'Alpha Contribution Over Time'
    ],
    specs=[[{'type': 'pie'}, {'type': 'bar'}],
           [{'type': 'scatter'}, {'type': 'scatter'}]]
)

# 1. Pie chart
all_contributions = list(factor_contributions.values()) + [alpha_contrib, rf_contrib]
all_labels = list(factor_contributions.keys()) + ['Alpha', 'Risk-Free']

# Only positive contributions for pie
pos_mask = [c > 0 for c in all_contributions]
pie_values = [c if c > 0 else 0 for c in all_contributions]

fig.add_trace(go.Pie(
    labels=all_labels,
    values=[abs(c) for c in all_contributions],
    textinfo='label+percent',
    marker=dict(colors=px.colors.qualitative.Set2)
), row=1, col=1)

# 2. Bar chart
colors = ['lime' if c > 0 else 'red' for c in all_contributions]
fig.add_trace(go.Bar(
    x=all_labels,
    y=[c * 100 for c in all_contributions],
    marker_color=colors
), row=1, col=2)

# 3. Cumulative attribution
for factor in ['MKT', 'SMB', 'HML', 'MOM']:
    fig.add_trace(go.Scatter(
        x=cum_attribution.index,
        y=cum_attribution[factor] * 100,
        mode='lines',
        name=factor
    ), row=2, col=1)

fig.add_trace(go.Scatter(
    x=cum_attribution.index,
    y=cum_attribution['Portfolio'] * 100,
    mode='lines',
    name='Portfolio',
    line=dict(color='white', width=3)
), row=2, col=1)

# 4. Alpha over time
fig.add_trace(go.Scatter(
    x=cum_attribution.index,
    y=cum_attribution['Alpha'] * 100,
    mode='lines',
    fill='tozeroy',
    line=dict(color='lime', width=2),
    fillcolor='rgba(0,255,0,0.2)',
    name='Cumulative Alpha'
), row=2, col=2)
fig.add_hline(y=0, line_dash='dash', line_color='gray', row=2, col=2)

fig.update_layout(
    title=dict(text='Factor Performance Attribution Dashboard', font=dict(size=20)),
    template='plotly_dark',
    height=700,
    showlegend=True
)

fig.update_yaxes(title_text='Contribution (%)', row=1, col=2)
fig.update_yaxes(title_text='Cumulative Return (%)', row=2, col=1)
fig.update_yaxes(title_text='Cumulative Alpha (%)', row=2, col=2)

fig.show()


---

## Section 6: Factor Stress Testing & Scenario Analysis

### Analyst Perspective (Citadel Risk Management)

**Stress Testing:**

Estimate portfolio impact from extreme factor moves:
- Historical scenarios (2008 crisis, 2020 COVID, 2022 rate hikes)
- Hypothetical shocks (±2σ factor moves)
- Factor rotation scenarios

**Scenario P&L:**
$$\Delta P = \sum_{f=1}^{k} \beta_f \cdot \Delta F_f$$

This helps risk managers understand tail risks and factor concentration.

In [ ]:
# =============================================================================
# SECTION 6: FACTOR STRESS TESTING
# =============================================================================

print("\n" + "="*80)
print("FACTOR STRESS TESTING & SCENARIO ANALYSIS")
print("="*80)

# 6.1 Historical stress scenarios
print("\n[6.1] Historical Stress Scenarios")
print("-" * 50)

# Define historical scenarios (monthly factor returns during stress)
scenarios = {
    'GFC Crash (Oct 2008)': {'MKT': -0.20, 'SMB': -0.04, 'HML': 0.08, 'RMW': -0.03, 'CMA': 0.02, 'MOM': 0.15},
    'COVID Crash (Mar 2020)': {'MKT': -0.15, 'SMB': -0.06, 'HML': -0.12, 'RMW': 0.02, 'CMA': -0.01, 'MOM': -0.20},
    '2022 Rate Hikes': {'MKT': -0.10, 'SMB': 0.02, 'HML': 0.08, 'RMW': 0.03, 'CMA': 0.02, 'MOM': -0.10},
    'Tech Bubble Burst (2000)': {'MKT': -0.12, 'SMB': 0.10, 'HML': 0.15, 'RMW': 0.05, 'CMA': 0.03, 'MOM': -0.25},
    'Flash Crash (2010)': {'MKT': -0.08, 'SMB': -0.02, 'HML': 0.02, 'RMW': 0.01, 'CMA': 0.00, 'MOM': -0.05},
}

stress_results = []

for scenario_name, factor_shocks in scenarios.items():
    # Calculate portfolio impact
    pnl = sum(portfolio_betas[f] * factor_shocks[f] for f in factor_names)
    
    # Factor breakdown
    factor_impacts = {f: portfolio_betas[f] * factor_shocks[f] for f in factor_names}
    
    stress_results.append({
        'Scenario': scenario_name,
        'Portfolio Impact': pnl,
        **{f'{f}_Impact': factor_impacts[f] for f in factor_names}
    })
    
    print(f"\n     {scenario_name}:")
    print(f"       Portfolio P&L: {pnl*100:+.2f}%")
    for f in factor_names:
        print(f"         {f}: {factor_impacts[f]*100:+.2f}% (shock: {factor_shocks[f]*100:+.1f}%)")

stress_df = pd.DataFrame(stress_results)


In [ ]:
# =============================================================================
# HYPOTHETICAL SHOCK ANALYSIS
# =============================================================================

print("\n[6.2] Hypothetical Factor Shocks (±2σ)")
print("-" * 50)

# Factor volatilities (monthly)
factor_monthly_vol = factor_df[factor_names].std() * np.sqrt(21)

shock_results = []

for factor in factor_names:
    shock_2sd = 2 * factor_monthly_vol[factor]
    
    # Portfolio impact from 2σ shock
    impact_up = portfolio_betas[factor] * shock_2sd
    impact_down = portfolio_betas[factor] * (-shock_2sd)
    
    shock_results.append({
        'Factor': factor,
        'Beta': portfolio_betas[factor],
        '2σ Shock': shock_2sd,
        '+2σ Impact': impact_up,
        '-2σ Impact': impact_down
    })

shock_df = pd.DataFrame(shock_results)
print(shock_df.round(4).to_string(index=False))


In [ ]:
# =============================================================================
# STRESS TEST VISUALIZATION
# =============================================================================

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Portfolio Impact by Scenario',
        'Factor Shock Sensitivity',
        'Scenario Decomposition (COVID Crash)',
        'Worst Case Analysis'
    ]
)

# 1. Scenario impacts
fig.add_trace(go.Bar(
    x=stress_df['Scenario'],
    y=stress_df['Portfolio Impact'] * 100,
    marker_color=['red' if x < 0 else 'lime' for x in stress_df['Portfolio Impact']]
), row=1, col=1)

# 2. Factor sensitivity
fig.add_trace(go.Bar(
    x=shock_df['Factor'],
    y=shock_df['+2σ Impact'] * 100,
    name='+2σ',
    marker_color='lime'
), row=1, col=2)
fig.add_trace(go.Bar(
    x=shock_df['Factor'],
    y=shock_df['-2σ Impact'] * 100,
    name='-2σ',
    marker_color='red'
), row=1, col=2)

# 3. COVID decomposition
covid_row = stress_df[stress_df['Scenario'].str.contains('COVID')].iloc[0]
covid_impacts = [covid_row[f'{f}_Impact'] * 100 for f in factor_names]
colors = ['red' if x < 0 else 'lime' for x in covid_impacts]
fig.add_trace(go.Bar(
    x=factor_names,
    y=covid_impacts,
    marker_color=colors
), row=2, col=1)

# 4. Worst case
# Simultaneous adverse shocks
worst_case = sum(portfolio_betas[f] * (-2 * factor_monthly_vol[f]) 
                 if portfolio_betas[f] > 0 
                 else portfolio_betas[f] * (2 * factor_monthly_vol[f]) 
                 for f in factor_names)

var_95 = np.percentile(portfolio_returns, 5) * np.sqrt(21) * 100
var_99 = np.percentile(portfolio_returns, 1) * np.sqrt(21) * 100

fig.add_trace(go.Bar(
    x=['Worst Scenario', 'VaR 95%', 'VaR 99%'],
    y=[worst_case * 100, var_95, var_99],
    marker_color=['red', 'orange', 'magenta']
), row=2, col=2)

fig.update_layout(
    title=dict(text='Factor Stress Testing Dashboard', font=dict(size=20)),
    template='plotly_dark',
    height=700,
    barmode='group'
)

fig.update_yaxes(title_text='Portfolio Impact (%)', row=1, col=1)
fig.update_yaxes(title_text='Impact (%)', row=1, col=2)
fig.update_yaxes(title_text='Impact (%)', row=2, col=1)
fig.update_yaxes(title_text='Loss (%)', row=2, col=2)

fig.show()


---

## Section 7: Summary & Risk Report

### Institutional Risk Report Format

In [ ]:
# =============================================================================
# SECTION 7: EXECUTIVE RISK SUMMARY
# =============================================================================

print("\n" + "="*80)
print("EXECUTIVE RISK SUMMARY")
print("="*80)

print(f"""
┌──────────────────────────────────────────────────────────────────────────────┐
│  FACTOR RISK ATTRIBUTION REPORT                                              │
│  Report Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}                                             │
├──────────────────────────────────────────────────────────────────────────────┤
│  PORTFOLIO OVERVIEW                                                          │
│    Holdings:           {len(symbols):>5} securities                                        │
│    Total Weight:       {weights.sum()*100:>5.1f}%                                              │
│    Analysis Period:    2 Years (504 trading days)                            │
├──────────────────────────────────────────────────────────────────────────────┤
│  RISK METRICS                                                                │
│    Total Volatility:   {total_vol*100:>6.2f}% (annualized)                                │
│    Factor Risk:        {np.sqrt(factor_variance)*100:>6.2f}% ({factor_risk_pct:.1f}% of total)                          │
│    Idiosyncratic:      {np.sqrt(idio_variance)*100:>6.2f}% ({idio_risk_pct:.1f}% of total)                          │
├──────────────────────────────────────────────────────────────────────────────┤
│  FACTOR EXPOSURES (BETAS)                                                    │
│    Market (MKT):       {portfolio_betas['MKT']:>+6.3f}                                            │
│    Size (SMB):         {portfolio_betas['SMB']:>+6.3f}                                            │
│    Value (HML):        {portfolio_betas['HML']:>+6.3f}                                            │
│    Profitability:      {portfolio_betas['RMW']:>+6.3f}                                            │
│    Investment:         {portfolio_betas['CMA']:>+6.3f}                                            │
│    Momentum:           {portfolio_betas['MOM']:>+6.3f}                                            │
├──────────────────────────────────────────────────────────────────────────────┤
│  PERFORMANCE ATTRIBUTION                                                     │
│    Total Return:       {total_return*100:>+6.2f}%                                             │
│    Factor Return:      {total_factor_contrib*100:>+6.2f}%                                             │
│    Alpha:              {alpha_contrib*100:>+6.2f}%                                             │
├──────────────────────────────────────────────────────────────────────────────┤
│  STRESS TEST RESULTS (Worst Case)                                            │
│    COVID-style shock:  {stress_df[stress_df['Scenario'].str.contains('COVID')]['Portfolio Impact'].iloc[0]*100:>+6.2f}%                                             │
│    2σ adverse:         {worst_case*100:>+6.2f}%                                             │
│    VaR (95%):          {var_95:>+6.2f}%                                             │
└──────────────────────────────────────────────────────────────────────────────┘

KEY OBSERVATIONS:
  • Portfolio has {portfolio_betas['MKT']:.2f}x market exposure (beta)
  • {'Tilted toward large-cap' if portfolio_betas['SMB'] < 0 else 'Tilted toward small-cap'} stocks (SMB: {portfolio_betas['SMB']:.3f})
  • {'Growth-oriented' if portfolio_betas['HML'] < 0 else 'Value-oriented'} portfolio (HML: {portfolio_betas['HML']:.3f})
  • Factor risk accounts for {factor_risk_pct:.1f}% of total variance
  • Primary risk driver: Market factor ({np.sqrt(factor_risk_contrib['MKT'])*100:.2f}% vol contribution)

RECOMMENDATIONS:
  1. Consider hedging market beta if expecting volatility
  2. Monitor momentum exposure given recent factor reversals
  3. Diversification benefit from idiosyncratic risk is {'limited' if idio_risk_pct < 20 else 'moderate'}
""")

print("\n" + "="*80)
print("END OF FACTOR RISK ATTRIBUTION REPORT")
print("="*80)


---

## Summary & Key Takeaways

This notebook demonstrated **institutional-grade factor analysis** methodology:

### Factor Analysis Framework
1. **Factor Exposure Analysis** - Fama-French regression with 6 factors
2. **Risk Decomposition** - Factor vs idiosyncratic risk
3. **Rolling Analysis** - Time-varying factor betas
4. **Performance Attribution** - Return decomposition by factor
5. **Stress Testing** - Historical and hypothetical scenarios

### Domain APIs Used
- `qj.eod.get_historical_prices`
- `qj.ff.get_factors`
- `local pandas portfolio weights`

### Best Practices for Professional Factor Analysis
- Use at least 2 years of data for stable beta estimates
- Track rolling exposures to detect regime changes
- Decompose risk into systematic vs idiosyncratic
- Stress test with historical and hypothetical scenarios
- Report factor attribution alongside performance